# MAIW v2 — Getting Started

**Multi-Agent Intelligent Warehouse — v2 Governed Agentic Operations**

This notebook walks through the canonical 17-step MAIW v2 developer journey:

1. Validate environment (Python, GPU, packages)
2. Install MAIW packages
3. Configure API keys and model endpoints
4. Generate Warehouse World (DataPack)
5. Validate DataPack integrity
6. Start backend API server
7. Start frontend UI
8. Open WORLD view (operational graph)
9. Activate Wave 17 at-risk scenario
10. Ask Copilot about the scenario
11. Inspect OperationalContextSnapshot
12. Run SOP-driven agent (OperationsCoordinationAgent)
13. Observe Deep Agents delegation
14. Review RecommendedAction
15. Govern / approve the recommendation
16. Observe LIVE outcome
17. Open Model Gateway Evaluation Lab

---

> **Prerequisites:** Python 3.10+, Node.js 20+, NVIDIA API key (`nvapi-...`)
> 
> **Time to complete:** ~30 minutes for first run, ~5 minutes on subsequent runs
> 
> **No database required:** Demo Mode uses a fully local DataPack — no PostgreSQL, Redis, Milvus, or Kafka.


## Step 1 — Validate Environment


In [ ]:
import sys
import subprocess

print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"

# Check Node.js
result = subprocess.run(["node", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"Node.js version: {result.stdout.strip()}")
else:
    print("WARNING: Node.js not found — frontend UI will not be available")

# Check GPU (optional — required for local NIM)
try:
    import subprocess
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    if gpu.returncode == 0:
        print(f"GPU: {gpu.stdout.strip()}")
    else:
        print("No GPU detected — using NVIDIA-hosted inference (requires NVIDIA API key)")
except FileNotFoundError:
    print("No GPU detected — using NVIDIA-hosted inference (requires NVIDIA API key)")

print("\nEnvironment check complete.")


## Step 2 — Install MAIW Packages

MAIW uses a monorepo with editable installs. Run once after cloning.


In [ ]:
import subprocess, sys

packages = [
    "packages/maiw-contracts",
    "packages/maiw-state",
    "packages/maiw-world",
    "packages/maiw-models",
    "packages/maiw-decision",
    "packages/maiw-mcp",
    "packages/maiw-skills",
    "packages/maiw-agents",
    "packages/maiw-execution",
    "apps/api",
]

# Install from repo root
import pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "packages").exists():
    # Try parent — notebook may be in notebooks/ subdirectory
    REPO_ROOT = REPO_ROOT.parent

print(f"Repo root: {REPO_ROOT}")
assert (REPO_ROOT / "packages").exists(), f"Could not find packages/ — run from repo root. cwd={REPO_ROOT}"

for pkg in packages:
    pkg_path = REPO_ROOT / pkg
    if pkg_path.exists():
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", str(pkg_path), "-q"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"  OK  {pkg}")
        else:
            print(f"  FAIL {pkg}: {result.stderr[:200]}")
    else:
        print(f"  SKIP {pkg} (not found)")

print("\nPackage install complete.")


## Step 3 — Configure API Keys and Model Endpoints

Copy `.env.example` to `.env` and set your NVIDIA API key.
For local NIM: set `MAIW_NIM_BASE_URL` to your NIM endpoint.


In [ ]:
import os, pathlib

# Locate repo root
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "packages").exists():
    REPO_ROOT = REPO_ROOT.parent

env_file = REPO_ROOT / ".env"
env_example = REPO_ROOT / ".env.example"

if env_file.exists():
    # Load .env into environment
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, val = line.partition("=")
            os.environ.setdefault(key.strip(), val.strip())
    print("Loaded .env")
else:
    print(f"No .env found at {env_file}")
    if env_example.exists():
        print(f"Copy {env_example} to {env_file} and set NVIDIA_API_KEY")
    else:
        print("Set NVIDIA_API_KEY environment variable manually")

# Verify key variable
api_key = os.environ.get("NVIDIA_API_KEY", "")
if api_key and api_key.startswith("nvapi-"):
    print(f"NVIDIA_API_KEY: {api_key[:12]}...{api_key[-4:]} (set)")
else:
    print("WARNING: NVIDIA_API_KEY not set or invalid — hosted inference will not work")

nim_url = os.environ.get("MAIW_NIM_BASE_URL", "")
print(f"MAIW_NIM_BASE_URL: {nim_url or '(not set — will use hosted NIM)'}")
print(f"MAIW_AGENT_RUNTIME: {os.environ.get('MAIW_AGENT_RUNTIME', 'deterministic (default)')}")


## Step 4 — Generate Warehouse World (DataPack)

The Warehouse World is a deterministic, seed-based synthetic warehouse environment.
The generator produces an immutable `WarehouseDataPack` on disk — all agents,
scenarios, and demos run against this same DataPack.


In [ ]:
import pathlib, os, sys

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "packages").exists():
    REPO_ROOT = REPO_ROOT.parent

# Add repo root to path for package resolution
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from maiw_world.generator import WarehouseWorldGenerator
from maiw_world.config import WarehouseWorldConfig

PACK_DIR = REPO_ROOT / "data" / "world" / "demo"
PACK_DIR.mkdir(parents=True, exist_ok=True)

config = WarehouseWorldConfig(seed=42, preset="dc47_demo")
gen = WarehouseWorldGenerator(config)

pack_path = PACK_DIR / "datapack.json"
if pack_path.exists():
    print(f"DataPack already exists: {pack_path}")
    print("Delete it to regenerate. Using existing DataPack.")
else:
    print("Generating DataPack (this takes ~5-10 seconds)...")
    pack = gen.generate()
    gen.save(pack, PACK_DIR)
    print(f"DataPack saved to: {PACK_DIR}")

print("\nWarehouse World generation complete.")


## Step 5 — Validate DataPack Integrity

Verifies: no duplicate IDs, no dangling edges, valid worker-task assignments,
wave/order consistency, and equipment/location validity.


In [ ]:
from maiw_world.datapack import WarehouseDataPack

pack = WarehouseDataPack.load(PACK_DIR)
print(f"DataPack ID: {pack.pack_id}")
print(f"Checksum:    {pack.checksum}")
print(f"Generated:   {pack.generated_at}")
print(f"Seed:        {pack.config.seed}")
print()

# Entity summary
print("Entity counts:")
for entity_type, entities in pack.entities.items():
    print(f"  {entity_type}: {len(entities)}")

print()
print(f"Relationships: {len(pack.relationships)}")
print(f"Events:        {len(pack.events)}")
print()

# Validate
issues = pack.validate()
if issues:
    for issue in issues:
        print(f"  ISSUE: {issue}")
    raise AssertionError(f"{len(issues)} DataPack integrity issues found")
else:
    print("DataPack integrity: OK — no issues found")


## Steps 6-7 — Start Backend and Frontend

Run these in separate terminal sessions (not inside the notebook).

**Backend:**
```bash
# From repo root
python -m uvicorn maiw_api.app:app --reload --port 8000
```

**Frontend:**
```bash
# From repo root
cd src/ui/web && npm install && npm run dev
# Runs at http://localhost:3000
```

Once running, the UI at `http://localhost:3000` shows:
- **WORLD** tab — Warehouse operational graph
- **COPILOT** tab — AI assistant
- **MODELS** tab — Model Gateway Lab


In [ ]:
# Verify the backend API is reachable
import urllib.request, json

API_BASE = "http://localhost:8000"

try:
    with urllib.request.urlopen(f"{API_BASE}/health", timeout=3) as resp:
        data = json.loads(resp.read())
        print(f"Backend status: {data.get('status', 'unknown')}")
        print(f"World loaded:   {data.get('world_loaded', 'unknown')}")
except Exception as e:
    print(f"Backend not reachable at {API_BASE}: {e}")
    print("Start the backend first (see Step 6 above).")


## Steps 8-9 — Open WORLD View and Activate Wave 17 Scenario


In [ ]:
import urllib.request, json, pprint

API_BASE = "http://localhost:8000"

# Get world summary
try:
    with urllib.request.urlopen(f"{API_BASE}/api/v1/world/summary", timeout=5) as resp:
        summary = json.loads(resp.read())
        print("Warehouse World Summary:")
        print(f"  Waves:     {summary.get('wave_count', 'N/A')}")
        print(f"  Workers:   {summary.get('worker_count', 'N/A')}")
        print(f"  Equipment: {summary.get('equipment_count', 'N/A')}")
        print(f"  State:     {summary.get('current_state', 'N/A')}")
except Exception as e:
    print(f"Could not reach world summary: {e}")
    print("Ensure backend is running (Step 6).")


In [ ]:
import urllib.request, json, urllib.parse

API_BASE = "http://localhost:8000"

# Activate the Wave 17 at-risk scenario
scenario_payload = json.dumps({"scenario_id": "wave_17_at_risk"}).encode()
req = urllib.request.Request(
    f"{API_BASE}/api/v1/demo/scenario/activate",
    data=scenario_payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, timeout=10) as resp:
        result = json.loads(resp.read())
        print(f"Scenario activated: {result.get('scenario_id', 'unknown')}")
        print(f"Description: {result.get('description', 'N/A')}")
        print(f"Affected entities: {result.get('affected_entity_count', 'N/A')}")
except Exception as e:
    print(f"Could not activate scenario: {e}")
    print("Ensure backend is running (Step 6).")


## Step 10 — Ask Copilot About the Scenario

The Copilot ASK/ANALYZE intents query the agent using the current WarehouseState
as operational context. No state is invented — every answer is grounded in the DataPack.


In [ ]:
import urllib.request, json, time

API_BASE = "http://localhost:8000"
CONVERSATION_ID = f"notebook-demo-{int(time.time())}"

# ASK: What is the current wave status?
payload = json.dumps({
    "prompt": "What is the status of Wave 17? Is it at risk of missing the carrier cutoff?",
    "conversation_id": CONVERSATION_ID,
    "turn_index": 1
}).encode()

req = urllib.request.Request(
    f"{API_BASE}/api/v1/copilot/turn",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, timeout=30) as resp:
        result = json.loads(resp.read())
        print("Copilot response:")
        print(f"  Intent:    {result.get('intent', 'N/A')}")
        print(f"  Answer:\n{result.get('answer', result.get('response', 'N/A'))}")
        print(f"  Turn ID:   {result.get('turn_id', 'N/A')}")
except Exception as e:
    print(f"Copilot call failed: {e}")
    print("Ensure backend is running with a valid NVIDIA_API_KEY.")


## Step 11 — Inspect OperationalContextSnapshot

The `OperationalContextSnapshot` is captured before every model call.
It is immutable — LIVE changes do not retroactively alter it.


In [ ]:
import urllib.request, json

API_BASE = "http://localhost:8000"

try:
    with urllib.request.urlopen(
        f"{API_BASE}/api/v1/world/context-snapshot?focus=wave&focus_id=wave-17",
        timeout=10
    ) as resp:
        snapshot = json.loads(resp.read())
        print("OperationalContextSnapshot:")
        print(f"  snapshot_id:   {snapshot.get('snapshot_id', 'N/A')}")
        print(f"  timestamp:     {snapshot.get('timestamp', 'N/A')}")
        print(f"  focus_entity:  {snapshot.get('focus_entity', 'N/A')}")
        print(f"  entity_count:  {len(snapshot.get('entities', {}))}")
        print(f"  checksum:      {snapshot.get('checksum', 'N/A')}")
except Exception as e:
    print(f"Context snapshot endpoint not reachable: {e}")


## Step 12 — Run SOP-Driven Agent

Trigger the `OperationsCoordinationAgent` with the `wave_risk_resolution` SOP.
This demonstrates the full SOP execution path with governance handoff.


In [ ]:
import urllib.request, json

API_BASE = "http://localhost:8000"

# Trigger agent via Copilot ACT intent
payload = json.dumps({
    "prompt": "Resolve the Wave 17 risk. Assess the situation and recommend the best action.",
    "conversation_id": CONVERSATION_ID,
    "turn_index": 2
}).encode()

req = urllib.request.Request(
    f"{API_BASE}/api/v1/copilot/turn",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.loads(resp.read())
        print(f"Intent: {result.get('intent', 'N/A')}")
        print(f"Status: {result.get('governance_status', result.get('status', 'N/A'))}")
        if result.get('recommendation'):
            rec = result['recommendation']
            print(f"\nRecommendation:")
            print(f"  capability: {rec.get('capability', 'N/A')}")
            print(f"  priority:   {rec.get('priority', 'N/A')}")
            print(f"  rationale:  {rec.get('rationale', 'N/A')[:200]}")
        if result.get('proposal_id'):
            print(f"\nProposal ID: {result['proposal_id']} (awaiting governance)")
            PROPOSAL_ID = result['proposal_id']
        else:
            PROPOSAL_ID = None
except Exception as e:
    print(f"Agent run failed: {e}")
    PROPOSAL_ID = None


## Step 13 — Observe Deep Agents Delegation

If `MAIW_AGENT_RUNTIME=deep_agents`, the OperationsCoordinationAgent delegates
to specialist subagents (LaborAgent, WaveAgent, EquipmentAgent) via Deep Agents.
Check the trace to see delegation events.


In [ ]:
import os

runtime = os.environ.get("MAIW_AGENT_RUNTIME", "deterministic")
print(f"Current agent runtime: {runtime}")

if runtime == "deep_agents":
    print("\nDeep Agents runtime active:")
    print("  - OperationsCoordinationAgent delegates to LaborAgent, WaveAgent")
    print("  - Each subagent runs in isolated mode (no parent context leakage)")
    print("  - All model calls go through ModelGateway")
    print("  - WRITE capabilities are hard-blocked")
    print("  - Result: RecommendedAction → WAITING_FOR_GOVERNANCE")
else:
    print("\nDeterministic runtime active (default):")
    print("  - SOP steps executed in exact order")
    print("  - No adaptive delegation")
    print("")
    print("To use Deep Agents: set MAIW_AGENT_RUNTIME=deep_agents in .env")
    print("and restart the backend.")


## Step 14 — Review RecommendedAction

The SOP output is a `RecommendedAction` — a semantic description of the best intervention.
It does NOT contain MCP parameters. The governance layer translates it into an `ActionProposal`.


In [ ]:
import urllib.request, json

API_BASE = "http://localhost:8000"

# Get the pending proposal if we have one
if PROPOSAL_ID:
    try:
        with urllib.request.urlopen(
            f"{API_BASE}/api/v1/governance/proposals/{PROPOSAL_ID}",
            timeout=10
        ) as resp:
            proposal = json.loads(resp.read())
            print(f"Proposal ID:     {proposal.get('proposal_id', 'N/A')}")
            print(f"Status:          {proposal.get('status', 'N/A')}")
            print(f"Capability:      {proposal.get('capability', 'N/A')}")
            print(f"Target:          {proposal.get('target_entity_id', 'N/A')}")
            print(f"Objective:       {proposal.get('objective', 'N/A')[:200]}")
            print(f"Rationale:       {proposal.get('rationale', 'N/A')[:200]}")
            print(f"Priority:        {proposal.get('priority', 'N/A')}")
            print(f"Expires:         {proposal.get('expires_at', 'N/A')}")
    except Exception as e:
        print(f"Could not fetch proposal {PROPOSAL_ID}: {e}")
else:
    print("No proposal ID from previous step — check Step 12 output.")


## Step 15 — Govern / Approve the Recommendation

**MAIW AUTHORITY BOUNDARY**

Approval is a human decision. In Demo Mode, the operator clicks APPROVE in the UI
(`http://localhost:3000` → Command Center → pending proposals).

For notebook demonstration, the API endpoint below simulates operator approval.
In production, only the governance route accepts approval decisions.


In [ ]:
import urllib.request, json

API_BASE = "http://localhost:8000"

if PROPOSAL_ID:
    payload = json.dumps({
        "decision": "APPROVED",
        "rationale": "Operator approved via notebook demo",
        "operator_id": "notebook-demo"
    }).encode()

    req = urllib.request.Request(
        f"{API_BASE}/api/v1/governance/proposals/{PROPOSAL_ID}/decision",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            result = json.loads(resp.read())
            print(f"Decision outcome: {result.get('decision_outcome', 'N/A')}")
            print(f"Execution status: {result.get('execution_status', 'N/A')}")
            print(f"Execution ID:     {result.get('execution_id', 'N/A')}")
    except Exception as e:
        print(f"Governance decision failed: {e}")
        print("The UI governance panel is the primary approval interface.")
else:
    print("No proposal ID — complete Step 12 first.")
    print("\nAlternatively, use the UI to approve pending proposals:")
    print("  http://localhost:3000 → Command Center → Pending Proposals")


## Step 16 — Observe LIVE Outcome

After execution, the Warehouse State is updated. The agent can observe the outcome
and determine whether the objective was met.


In [ ]:
import urllib.request, json

API_BASE = "http://localhost:8000"

# Check updated wave status
try:
    with urllib.request.urlopen(
        f"{API_BASE}/api/v1/world/live/wave/wave-17",
        timeout=10
    ) as resp:
        live_state = json.loads(resp.read())
        print("Wave 17 LIVE state:")
        print(f"  Status:         {live_state.get('status', 'N/A')}")
        print(f"  Risk level:     {live_state.get('risk_level', 'N/A')}")
        print(f"  At-risk tasks:  {live_state.get('at_risk_count', 'N/A')}")
        print(f"  Pending tasks:  {live_state.get('pending_count', 'N/A')}")
        print(f"  State source:   {live_state.get('state_source', 'N/A')}")
except Exception as e:
    print(f"LIVE state endpoint not reachable: {e}")

# Ask Copilot for post-execution assessment
payload = json.dumps({
    "prompt": "What is the current status of Wave 17 after the intervention?",
    "conversation_id": CONVERSATION_ID,
    "turn_index": 3
}).encode()

req = urllib.request.Request(
    f"{API_BASE}/api/v1/copilot/turn",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, timeout=30) as resp:
        result = json.loads(resp.read())
        print(f"\nPost-execution Copilot assessment:")
        print(result.get('answer', result.get('response', 'N/A')))
except Exception as e:
    print(f"Post-execution Copilot assessment failed: {e}")


## Step 17 — Open Model Gateway Evaluation Lab

The Model Gateway Evaluation Lab is a read-only developer tool at `/models/lab`.
It shows pre-computed evaluation artifacts from the ModelGateway benchmark runs.

Constraints:
- Read-only — cannot create proposals, invoke DecisionEngine, or call write MCP
- Artifact paths are whitelisted — no path traversal possible
- Secrets stripped from all responses


In [ ]:
import urllib.request, json

API_BASE = "http://localhost:8000"

# List available Model Lab runs
try:
    with urllib.request.urlopen(
        f"{API_BASE}/api/v1/models/lab/runs",
        timeout=10
    ) as resp:
        runs = json.loads(resp.read())
        print("Model Gateway Evaluation Lab — available runs:")
        for run in runs.get('runs', []):
            print(f"  {run.get('run_id')}: {run.get('name')} ({run.get('case_count', '?')} cases)")
        print(f"\nOpen in browser: http://localhost:3000/models")
except Exception as e:
    print(f"Model Lab endpoint not reachable: {e}")
    print("Ensure backend is running.")


## Summary

You have completed the MAIW v2 canonical developer journey:

| Step | What you did |
|------|--------------|
| 1    | Validated Python, GPU, Node.js environment |
| 2    | Installed all MAIW packages (editable install) |
| 3    | Configured NVIDIA API key and model endpoints |
| 4    | Generated deterministic Warehouse World (DataPack, seed=42) |
| 5    | Validated DataPack integrity (0 issues) |
| 6-7  | Started backend (uvicorn) and frontend (npm) |
| 8    | Opened WORLD view (operational graph) |
| 9    | Activated Wave 17 at-risk scenario |
| 10   | Asked Copilot for wave status assessment |
| 11   | Inspected OperationalContextSnapshot (immutable, checksummed) |
| 12   | Ran OperationsCoordinationAgent with wave_risk_resolution SOP |
| 13   | Observed Deep Agents / Deterministic runtime delegation |
| 14   | Reviewed RecommendedAction (semantic intent, no MCP params) |
| 15   | Approved via governance (MAIW authority boundary) |
| 16   | Observed LIVE warehouse outcome |
| 17   | Explored Model Gateway Evaluation Lab |

---

**Next steps:**
- [Adding an Agent or Skill](../docs/developer/ADDING_AN_AGENT_OR_SKILL.md)
- [SOP Development Guide](../docs/developer/SOP_DEVELOPMENT.md) *(coming soon)*
- [Model Gateway Evaluation](../docs/developer/MODEL_GATEWAY_EVALUATION.md)
- [Architecture Overview](../docs/architecture/AGENT_RUNTIME.md)
- [MAIW Glossary](../docs/GLOSSARY.md)
